# 05b Quantile CNN fix

The first version of the quantile CNN made its quantiles non-crossing by adding softplus gaps to the median. Several gaps stopped learning, so the 25%, 50% and 75% quantiles collapsed together (on select days, 39 to 52% of outcomes fell below the 25% quantile). This version sorts the nine raw outputs instead; inputs, loss, seeds, split and early stopping are unchanged.

The old checkpoints and predictions are kept in `review/quantile_v1/`, and the change is recorded in `review/deviations.md`. Notebook 09 has to be re-run afterwards.

In [ ]:
!pip install -q datasets pvlib
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/UROP'
LOCAL_CACHE = '/content/skippd_cache'

In [ ]:
import os, shutil, time, math, json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
P_DIR = f'{OUT}/review'
for d in [P_DIR, f'{P_DIR}/ckpt', f'{P_DIR}/preds', f'{P_DIR}/figs', f'{OUT}/cache', LOCAL_CACHE]:
    os.makedirs(d, exist_ok=True)
SEED = 42
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

In [ ]:
INPUT_LEN, HORIZON = 15, 15
LAT, LON, ALT = 37.4275, -122.1697, 23
PANEL_TILT, PANEL_AZIMUTH, PANEL_KW_DC = 22.5, 195, 30.1
PV_CAP_PCT = 99.95
SEL_C = dict(geo=True, cs=True, kout=True, var=True, resid=True)
TRAIN = dict(batch=256, lr=1e-3, max_epochs=25, patience=4)
DIFF = dict(T=100, max_epochs=60, patience=6, lr=2e-4, batch=512, n_samples=50)
SPLIT_FRAC = dict(train=0.70, select=0.15, calib=0.15)
TIME_TOL_S = 1.0
SPIKE_KW = 8.0
QUANTILES = [0.025, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.975]

In [ ]:
def load_split(split):
    names = ['img', 'pv', 'ts']
    local = {k: f'{LOCAL_CACHE}/{split}_{k}.npy' for k in names}
    drive_ = {k: f'{OUT}/cache/{split}_{k}.npy' for k in names}
    if not all(os.path.exists(p) for p in local.values()) and all(os.path.exists(p) for p in drive_.values()):
        print(f'[{split}] copying cache from Drive to local disk...')
        for k in names: shutil.copy(drive_[k], local[k])
    if all(os.path.exists(p) for p in local.values()):
        print(f'[{split}] loading from cache')
        img = np.load(local['img']); pv = np.load(local['pv']); ts_raw = np.load(local['ts'])
    else:
        from datasets import load_dataset
        print(f'[{split}] downloading from Hugging Face (first run only)...')
        ds = load_dataset('solarbench/SKIPPD', split=split)
        n = len(ds); img = np.empty((n, 64, 64, 3), dtype=np.uint8); pos = 0
        for b in ds.iter(batch_size=2000):
            arr = np.stack([np.asarray(im.convert('RGB')) for im in b['image']]); img[pos:pos + len(arr)] = arr; pos += len(arr)
        pv = np.asarray(ds['pv'], dtype=np.float32)
        t = pd.to_datetime(ds['time'])
        if t.tz is not None: t = t.tz_convert('UTC').tz_localize(None)
        ts_raw = t.values.astype('datetime64[ns]')
        for k, v in [('img', img), ('pv', pv), ('ts', ts_raw)]:
            np.save(local[k], v); shutil.copy(local[k], drive_[k])
    order = np.argsort(ts_raw, kind='stable')
    return dict(name=split, img=img, img_idx=order, pv=pv[order].astype(np.float32), ts=pd.DatetimeIndex(ts_raw[order]))

In [ ]:
TR = load_split('train'); TE = load_split('test')
P = np.load(f'{OUT}/day1b_prep.npz', allow_pickle=True)
TR['good'], TE['good'] = P['tr_good'].astype(bool), P['te_good'].astype(bool)
print('train rows', len(TR['pv']), '| test rows', len(TE['pv']))

In [ ]:
from pvlib.location import Location
from pvlib import irradiance
SITE = Location(LAT, LON, altitude=ALT)
def solar_terms(ts_naive):
    a = pd.DatetimeIndex(ts_naive).tz_localize('UTC')
    cs = SITE.get_clearsky(a, model='ineichen'); sp = SITE.get_solarposition(a)
    poa = irradiance.get_total_irradiance(PANEL_TILT, PANEL_AZIMUTH, dni=cs['dni'], ghi=cs['ghi'], dhi=cs['dhi'],
                                          solar_zenith=sp['apparent_zenith'], solar_azimuth=sp['azimuth'])
    return (np.clip(PANEL_KW_DC * poa['poa_global'].values / 1000.0, 0, PANEL_KW_DC).astype(np.float32),
            sp['apparent_zenith'].values.astype(np.float32), sp['azimuth'].values.astype(np.float32))
def local_date(ts): return np.array((pd.DatetimeIndex(ts) - pd.Timedelta(hours=8)).date)
def is_dst(d): return bool((pd.Timestamp(d) + pd.Timedelta(hours=12)).tz_localize('America/Los_Angeles').dst())
CLOCK_FIX = json.load(open(f'{OUT}/clock_fix.json')); print('clock fix:', CLOCK_FIX)
def solar_time(ts):
    ts = pd.DatetimeIndex(ts); ld = local_date(ts); lut = {d: is_dst(d) for d in set(ld)}
    shift = np.where(np.array([lut[d] for d in ld]), CLOCK_FIX['dst_min'], CLOCK_FIX['std_min'])
    return ts - pd.to_timedelta(shift, unit='min')
for D in (TR, TE):
    cs, zen, azi = solar_terms(solar_time(D['ts']))
    D['cs_raw'] = np.nan_to_num(cs); D['zen'] = np.nan_to_num(zen, nan=180.0); D['azi'] = np.nan_to_num(azi)
    D['ld'] = local_date(D['ts'])
    doy = D['ts'].dayofyear.values
    D['doy_s'] = np.sin(2 * np.pi * doy / 365.25).astype(np.float32); D['doy_c'] = np.cos(2 * np.pi * doy / 365.25).astype(np.float32)

In [ ]:
def spike_flags(pv):
    d1 = np.r_[0, np.diff(pv)]; d2 = np.r_[np.diff(pv), 0]
    return (np.abs(d1) > SPIKE_KW) & (np.abs(d2) > SPIKE_KW) & (np.sign(d1) != np.sign(d2))

def build_windows(D, good, exact=True):
    """Windows of INPUT_LEN readings plus a target HORIZON readings later.
       exact=True: every step must be 60 s +/- TIME_TOL_S (so the target is exactly 15 min after issue).
       exact=False: the notebook 01 rule (no step > 1 min, no step <= 0)."""
    s = D['ts'].values.astype('datetime64[ns]').astype(np.int64) / 1e9
    d = np.diff(s)
    bad = (np.abs(d - 60.0) > TIME_TOL_S) if exact else ((d > 60.0 + 1e-6) | (d <= 0))
    cb = np.r_[0, np.cumsum(bad)]
    t = np.arange(INPUT_LEN - 1, len(s) - HORIZON); st, en = t - (INPUT_LEN - 1), t + HORIZON
    ok = (cb[en] - cb[st]) == 0
    cg = np.r_[0, np.cumsum(~good)]; ok &= (cg[en + 1] - cg[st]) == 0
    return t[ok]

def horizon_seconds(D, w):
    s = D['ts'].values.astype('datetime64[ns]').astype(np.int64) / 1e9
    return s[w + HORIZON] - s[w]

In [ ]:
Z = np.load(f'{P_DIR}/windows_and_splits.npz', allow_pickle=True)
W = {k[2:]: Z[k] for k in Z.files if k.startswith('w_')}
DAY_SPLIT = dict(zip(pd.to_datetime(Z['tr_days']).date, Z['tr_day_split']))
TR['split'] = np.array([DAY_SPLIT[d] for d in TR['ld']])
PV_CAP = float(Z['pv_cap'])
for D in (TR, TE):
    D['cs'] = np.minimum(D['cs_raw'], PV_CAP).astype(np.float32)
    D['k'] = np.clip(D['pv'] / np.maximum(D['cs'], 1.0), 0, 1.5).astype(np.float32)
    D['spike'] = spike_flags(D['pv'])
for k, v in W.items(): print(f'{k:>10}: {len(v)} windows')
print(f'PV cap (training days): {PV_CAP:.2f} kW')

In [ ]:
OFF = np.arange(-(INPUT_LEN - 1), 1)
def smart_base(D, t):
    return np.clip(D['pv'][t] / np.maximum(D['cs_raw'][t], 1e-3) * D['cs_raw'][t + HORIZON], 0, PV_CAP).astype(np.float32)

def vec_feats(D, t, c):
    H = t[:, None] + OFF; tg = t + HORIZON
    parts = [D['pv'][H] / PV_CAP]
    if c.get('geo'):
        parts.append(np.stack([D['doy_s'][t], D['doy_c'][t], D['zen'][t] / 90, D['zen'][tg] / 90,
                               np.cos(np.radians(D['zen'][t])), np.sin(np.radians(D['azi'][t])), np.cos(np.radians(D['azi'][t]))], 1))
    if c.get('cs'): parts += [D['cs'][H] / PV_CAP, (D['cs'][tg] / PV_CAP)[:, None]]
    if c.get('kout'): parts.append(D['k'][H])
    if c.get('var'):
        kh, ph = D['k'][H], D['pv'][H]
        parts.append(np.stack([kh.std(1), np.abs(np.diff(ph, axis=1)).mean(1) / PV_CAP,
                               (ph[:, -1] - ph[:, -6]) / PV_CAP, (ph.max(1) - ph.min(1)) / PV_CAP], 1))
    if c.get('resid'): parts.append((smart_base(D, t) / PV_CAP)[:, None])
    return np.concatenate(parts, 1).astype(np.float32)

def to_img(D, t):
    return torch.from_numpy(D['img'][D['img_idx'][t]]).to(device).permute(0, 3, 1, 2).float().div_(255.0)

def batches(D, w, c, bs, shuffle, rng=None):
    idx = rng.permutation(len(w)) if shuffle else np.arange(len(w))
    for i in range(0, len(w), bs):
        t = w[idx[i:i + bs]]
        xv = torch.from_numpy(vec_feats(D, t, c)).to(device)
        img = to_img(D, t)
        y = torch.from_numpy(D['pv'][t + HORIZON] / PV_CAP).float().to(device)
        yield xv, img, y

N_VEC = vec_feats(TR, W['train'][:2], SEL_C).shape[1]

def conv_branch(img_ch=3):
    return nn.Sequential(nn.Conv2d(img_ch, 16, 5, stride=2, padding=2), nn.ReLU(),
                         nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(),
                         nn.Conv2d(32, 32, 3, stride=2, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(4))

In [ ]:
def save_preds(name, seed, arrays, meta=None):
    p = f'{P_DIR}/preds/{name}_seed{seed}.npz'
    np.savez_compressed(p, **arrays, meta=json.dumps(meta or {}))
    return p
def have_preds(name, seed): return os.path.exists(f'{P_DIR}/preds/{name}_seed{seed}.npz')
def load_preds(name, seed): return dict(np.load(f'{P_DIR}/preds/{name}_seed{seed}.npz', allow_pickle=True))
EVAL_SETS = ['select', 'calib', 'test', 'test_broad']
def data_of(k): return TE if k.startswith('test') else TR

In [ ]:
class CorrNet(nn.Module):
    """Correction CNN with a configurable head. Output is in units of PV/PV_CAP.
       kind='mse'  : 1 output, forecast = smart persistence + correction
       kind='gauss': 2 outputs, mean = smart persistence + correction, variance = softplus(raw) + floor
       kind='quant': len(QUANTILES) outputs = smart persistence + sorted corrections (non-crossing)"""
    def __init__(self, kind, use_image=True, p_drop=0.2):
        super().__init__()
        self.kind, self.use_image = kind, use_image
        if use_image: self.conv = conv_branch()
        self.vec = nn.Sequential(nn.Linear(N_VEC, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU())
        n_out = {'mse': 1, 'gauss': 2, 'quant': len(QUANTILES)}[kind]
        self.head = nn.Sequential(nn.Linear((32 * 16 if use_image else 0) + 32, 64), nn.ReLU(), nn.Dropout(p_drop), nn.Linear(64, n_out))
    def forward(self, xv, img):
        f = self.vec(xv)
        if self.use_image: f = torch.cat([self.conv(img).flatten(1), f], 1)
        o = self.head(f); sp = xv[:, -1]
        if self.kind == 'mse': return sp + o[:, 0]
        if self.kind == 'gauss': return sp + o[:, 0], F.softplus(o[:, 1]) + 1e-6
        return sp[:, None] + torch.sort(o, 1).values

QT = torch.tensor(QUANTILES, device=device)
def loss_of(kind, out, y):
    if kind == 'mse': return F.mse_loss(out, y)
    if kind == 'gauss': mu, var = out; return F.gaussian_nll_loss(mu, y, var, full=False, eps=1e-6)
    e = y[:, None] - out; return torch.maximum(QT * e, (QT - 1) * e).mean()

def train_cnn(kind, seed, use_image=True):
    tag = f'{kind}{"" if use_image else "_noimg"}_seed{seed}'; ck = f'{P_DIR}/ckpt/p1_{tag}.pt'
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    model = CorrNet(kind, use_image).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=TRAIN['lr'], weight_decay=1e-4)
    st = dict(epoch=0, best=float('inf'), bad=0, hist=[], best_state=None, done=False, train_sec=0.0)
    if os.path.exists(ck):
        st = torch.load(ck, map_location=device, weights_only=False); model.load_state_dict(st['model']); opt.load_state_dict(st['opt'])
        if not st['done']: print(f'[{tag}] resuming from epoch {st["epoch"]}')
    while not st['done']:
        t0 = time.time(); model.train(); tl = []
        for xv, img, y in batches(TR, W['train'], SEL_C, TRAIN['batch'], True, rng):
            loss = loss_of(kind, model(xv, img), y); opt.zero_grad(); loss.backward(); opt.step(); tl.append(loss.item())
        model.eval(); vs, vn = 0.0, 0
        with torch.no_grad():
            for xv, img, y in batches(TR, W['select'], SEL_C, 2048, False):
                vs += loss_of(kind, model(xv, img), y).item() * len(y); vn += len(y)
        v = vs / vn; st['hist'].append((float(np.mean(tl)), v)); st['epoch'] += 1; st['train_sec'] += time.time() - t0
        if v < st['best']: st['best'], st['bad'] = v, 0; st['best_state'] = {k: x.detach().clone() for k, x in model.state_dict().items()}
        else: st['bad'] += 1
        st['done'] = st['epoch'] >= TRAIN['max_epochs'] or st['bad'] >= TRAIN['patience']
        st['model'], st['opt'] = model.state_dict(), opt.state_dict(); torch.save(st, ck)
        print(f'[{tag}] epoch {st["epoch"]:2d}  train {np.mean(tl):.5f}  select {v:.5f}  ({time.time()-t0:.0f}s)')
    model.load_state_dict(st['best_state']); model.eval()
    return model, st

@torch.no_grad()
def predict_cnn(model, D, w):
    outs = []
    for xv, img, _ in batches(D, w, SEL_C, 2048, False):
        o = model(xv, img)
        if model.kind == 'gauss': o = torch.stack([o[0], o[1].sqrt()], 1)
        elif model.kind == 'mse': o = o[:, None]
        outs.append(o.cpu().numpy())
    return np.concatenate(outs) * PV_CAP

In [ ]:
SEEDS = [0, 1, 2]
V1 = f'{P_DIR}/quantile_v1'; os.makedirs(V1, exist_ok=True)
for s in SEEDS:
    for src in [f'{P_DIR}/preds/cnn_quant_seed{s}.npz', f'{P_DIR}/ckpt/p1_quant_seed{s}.pt']:
        meta_ok = True
        if src.endswith('.npz') and os.path.exists(src):
            meta_ok = json.loads(str(np.load(src, allow_pickle=True)['meta'])).get('version') != 'v2'
        if os.path.exists(src) and meta_ok and not os.path.exists(f'{V1}/{os.path.basename(src)}'):
            shutil.move(src, f'{V1}/{os.path.basename(src)}'); print('moved', os.path.basename(src), '-> quantile_v1/')
print('v1 files kept in', V1, ':', sorted(os.listdir(V1)))

## Retraining

In [ ]:
COST = []
for s in SEEDS:
    if have_preds('cnn_quant', s): print(f'cnn_quant seed {s}: v2 predictions already saved, skipping'); continue
    m, st = train_cnn('quant', s)
    arr = {k: predict_cnn(m, data_of(k), W[k]) for k in EVAL_SETS}
    save_preds('cnn_quant', s, arr, dict(kind='quant', version='v2', seed=s, epochs=st['epoch'], best_select_loss=st['best'], train_sec=st['train_sec']))
    COST.append(dict(model='cnn_quant_v2', seed=s, epochs=st['epoch'], best_select_loss=st['best'], train_minutes=st['train_sec'] / 60))
    del m
    if device == 'cuda': torch.cuda.empty_cache()
if COST:
    pd.DataFrame(COST).to_csv(f'{P_DIR}/p1_training_cost.csv', mode='a', header=False, index=False); print(pd.DataFrame(COST).round(3).to_string(index=False))

## Calibration check on select days

In [ ]:
y_se = TR['pv'][W['select'] + HORIZON]; rows = []
for ver, path in [('v1', V1), ('v2', f'{P_DIR}/preds')]:
    for s in SEEDS:
        p = f'{path}/cnn_quant_seed{s}.npz'
        if not os.path.exists(p): continue
        Q = np.load(p, allow_pickle=True)['select']; ql = np.array(QUANTILES)
        pin = 2 * np.mean([np.mean(np.maximum(q * (y_se - Q[:, i]), (q - 1) * (y_se - Q[:, i]))) for i, q in enumerate(ql)])
        rows.append(dict(version=ver, seed=s, select_quantile_loss=pin, **{f'{q:g}': (y_se <= Q[:, i]).mean() for i, q in enumerate(ql)}))
CHK = pd.DataFrame(rows); print(CHK.round(3).to_string(index=False))
CHK.to_csv(f'{P_DIR}/p1b_quantile_check_select.csv', index=False)

In [ ]:
dev = f'{P_DIR}/deviations.md'
if not (os.path.exists(dev) and 'quantile CNN baseline retrained (v2)' in open(dev).read()):
  with open(dev, 'a') as fh:
    fh.write(f"\n- {pd.Timestamp.now().isoformat(timespec='seconds')}: quantile CNN baseline retrained (v2). "
             "v1 made quantiles non-crossing with cumulative softplus gaps; several gaps stopped learning and the 25-75% quantiles "
             "collapsed (diagnosed on select days). v2 sorts the raw outputs. Inputs, loss, seeds, split and early stopping unchanged. "
             "v1 files kept in review/quantile_v1/.\n")
print(open(f'{P_DIR}/deviations.md').read())
print('Done. Next: re-run notebook 09.')